In [0]:
%sql

CREATE OR REPLACE VIEW small_projects.finops.finops_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: small_projects.finops.gold_usage_cost
  comment: FinOps cost and usage metrics for the small_projects workspace
  dimensions:
    - name: Month
      expr: usage_month
    - name: Workspace
      expr: workspace_name
    - name: Team
      expr: tag_team
    - name: Resource
      expr: resource_name
    - name: Compute Type
      expr: sku_name
    - name: Environment
      expr: environment
  measures:
    - name: Total Cost
      expr: SUM(total_cost)
    - name: Idle Cost
      expr: SUM(idle_cost)
    - name: Idle Cost Share
      expr: MEASURE(`Idle Cost`) / MEASURE(`Total Cost`)
    - name: Total DBUs
      expr: SUM(dbus)
    - name: Cost per DBU
      expr: MEASURE(`Total Cost`) / MEASURE(`Total DBUs`)
    - name: Potential Savings
      expr: SUM(autoterm_saving) + SUM(jobcompute_saving)
$$;

SELECT
  `Month`,
  MEASURE(`Total Cost`) AS `Total Cost`,
  MEASURE(`Idle Cost Share`) AS `Idle Cost Share`,
  MEASURE(`Potential Savings`) AS `Potential Savings`
FROM small_projects.finops.finops_metrics
GROUP BY ALL
ORDER BY `Month` ASC;

